 # Parks OSM

In [ ]:
!pip install -q osmnx overpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.2/100.2 kB 4.1 MB/s eta 0:00:00


In [ ]:
import osmnx as ox
import geopandas as gpd
import folium

# Define area of interest and OSM tags
place = "Milano, Lombardia, Italia"
tags = {'leisure': ['park', 'garden']}

# Download park features from OSM
gdf_osm_parks = ox.features_from_place(place, tags)

# Keep only polygon geometries
gdf_osm_parks = gdf_osm_parks[gdf_osm_parks.geometry.type.isin(['Polygon', 'MultiPolygon'])]

# Project to meters for area calculation
gdf_osm_parks_m = gdf_osm_parks.to_crs(epsg=3857)
gdf_osm_parks_m["area_m2"] = gdf_osm_parks_m.geometry.area

# Filter parks with area ≥ 5000 m²
gdf_osm_parks_large = gdf_osm_parks_m[gdf_osm_parks_m["area_m2"] >= 5000]

# Reproject to EPSG:4326 for web mapping
gdf_osm_parks_large = gdf_osm_parks_large.to_crs(epsg=4326)

# Create folium map centered on centroid of all parks
center = gdf_osm_parks_large.unary_union.centroid
mappa_osm = folium.Map(location=[center.y, center.x], zoom_start=12)

# Add parks to the map
for _, row in gdf_osm_parks_large.iterrows():
    name = row.get("name", "Unnamed")
    area = row["area_m2"]
    folium.GeoJson(
        row.geometry,
        tooltip=f"<b>{name}</b><br>{area:.0f} m²",
        style_function=lambda x: {
            "fillColor": "green",
            "color": "darkgreen",
            "weight": 1,
            "fillOpacity": 0.4
        }
    ).add_to(mappa_osm)

folium.LayerControl().add_to(mappa_osm)
mappa_osm


<ipython-input-3-375a87805c78>:30: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = gdf_osm_parks_large.unary_union.centroid


In [ ]:
gdf_osm_parks_large.to_file("parks.gpkg", layer="parks", driver="GPKG")
